In [12]:
print("Assignment 3")

Assignment 3


In [13]:
import streamlit, plotly, subprocess
nmap_v = subprocess.run(['nmap','--version'], capture_output=True, text=True)

In [14]:
import pandas as pd

comparison = {
    'Approach': [
        'Hardcoded string',
        'Hardcoded string',
        'os.environ.get()',
        'Colab Secrets  (userdata.get)',
    ],
    'What the code contains': [
        'password = "mypassword123"',
        'api_key  = "abc123xyz"',
        'password = os.environ.get("GMAIL_PASSWORD")',
        'password = userdata.get("GMAIL_PASSWORD")',
    ],
    'Safe to share?': [
        'NO — value visible in file',
        'NO — key visible in file',
        'YES — value lives in environment',
        'YES — value lives in encrypted Colab storage',
    ]
}
pd.DataFrame(comparison)

,Approach,What the code contains,Safe to share?
0,Hardcoded string,"password = ""mypassword123""",NO — value visible in file
1,Hardcoded string,"api_key = ""abc123xyz""",NO — key visible in file
2,os.environ.get(),"password = os.environ.get(""GMAIL_PASSWORD"")",YES — value lives in environment
3,Colab Secrets (userdata.get),"password = userdata.get(""GMAIL_PASSWORD"")",YES — value lives in encrypted Colab storage


In [15]:
from dotenv import load_dotenv
import os

load_dotenv("u.env")  # here i will storing my keys  that i want  in a u.env file So that i will get the keys from that file 

VT_API_KEY = os.getenv("VT_API_KEY")
sender_email = os.getenv("GMAIL_SENDER")
app_password = os.getenv("GMAIL_PASSWORD") #Disclaimer : you have tosave the password in file don't use space in between them 
recipient_email = os.getenv("GMAIL_RECIPIENT")
targets_env = os.getenv("SCAN_TARGETS")

Check = ['VT_API_KEY','GMAIL_SENDER','GMAIL_PASSWORD','GMAIL_RECIPIENT','SCAN_TARGETS'] # here  to  check is that they are loaded are not
print("ENV VARIABLES STATUS")
all_ok = True

for s in Check:

    val = os.getenv(s)

    if val:
        status = "set"
    else:
        status = "missing"
        all_ok = False

    print(f"{s:<22} : {status}")

if all_ok:
    print("All environment variables are Loaded")
else:
    print("Once Check it is that loaded are not")



ENV VARIABLES STATUS
VT_API_KEY             : set
GMAIL_SENDER           : set
GMAIL_PASSWORD         : set
GMAIL_RECIPIENT        : set
SCAN_TARGETS           : set
All environment variables are Loaded


In [16]:
#Install libraries (run once)


import subprocess, sys

for pkg in ["streamlit","requests","pandas","plotly",
            "beautifulsoup4","python-dotenv"]:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q",
         "--disable-pip-version-check"],
        capture_output=True, text=True)
    status = "✅" if result.returncode == 0 else "❌"
    print(f"{status}  {pkg}")

print("\n✅  All done — run Cell 2 to write app.py!")


app_code = '''
import streamlit as st
import requests, pandas as pd, ssl, socket, os, smtplib, time
import plotly.express as px
import plotly.graph_objects as go
from bs4 import BeautifulSoup  #is used to read and understand the HTML code of a webpage in an easy way.
from datetime import datetime
from urllib.parse import urlparse
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from dotenv import load_dotenv   # for loading my env iam using this 

load_dotenv("u.env")
SENDER    = os.getenv("GMAIL_SENDER",    "")
PASSWORD  = os.getenv("GMAIL_PASSWORD",  "")
RECEIVER  = os.getenv("GMAIL_RECIPIENT", "")

st.set_page_config(page_title="CyberSentry", page_icon="\\U0001f6e1\\ufe0f",
                   layout="wide")

SEV_SCORE = {"Critical":10,"High":7,"Medium":4,"Low":2,"Informational":0}
SEV_COLOR = {"Critical":"#e53e3e","High":"#dd6b20","Medium":"#d69e2e",
             "Low":"#3182ce","Informational":"#718096"}
GRADE_COL = {"A":"#38a169","B":"#68d391","C":"#d69e2e","D":"#dd6b20","F":"#e53e3e"}
# this arfe the Targets 
TEST_SITES = [
    "http://testphp.vulnweb.com",
    "http://testasp.vulnweb.com",
    "http://testhtml5.vulnweb.com",
    "http://zero.webappsecurity.com",
    "Custom — type below",
]

# ── Safe HTTP GET helper ──────────────────────────────────────
def safe_get(url, timeout=8, redirects=True):
    try:
        r = requests.get(url, timeout=timeout, allow_redirects=redirects,
                         headers={"User-Agent":"CyberSentry-Scanner/1.0"})
        return r, None
    except requests.exceptions.SSLError        : return None, "SSL error"
    except requests.exceptions.ConnectionError : return None, "Cannot connect"
    except requests.exceptions.Timeout         : return None, "Timed out"
    except Exception as e                      : return None, str(e)

#Here iam Checking Vulnerability checks 

def chk_headers(resp):
    out, h = [], resp.headers
    for hdr, sev, msg, fix in [
        ("Strict-Transport-Security","Critical","HSTS missing",
         "Add: Strict-Transport-Security: max-age=31536000; includeSubDomains"),
        ("Content-Security-Policy","High","No Content-Security-Policy",
         "Set a CSP header to block XSS and injection attacks"),
        ("X-Frame-Options","High","X-Frame-Options missing — clickjacking risk",
         "Add X-Frame-Options: DENY"),
        ("X-Content-Type-Options","Medium","X-Content-Type-Options missing",
         "Add X-Content-Type-Options: nosniff"),
        ("Referrer-Policy","Low","Referrer-Policy not set",
         "Add Referrer-Policy: strict-origin-when-cross-origin"),
        ("Permissions-Policy","Low","Permissions-Policy missing",
         "Restrict camera/mic/location via Permissions-Policy header"),
    ]:
        if hdr not in h:
            out.append(("Security Headers", msg, sev, f"Header \'{hdr}\' absent", fix))
    return out

def chk_ssl(host):
    out = []
    try:
        ctx = ssl.create_default_context()
        with ctx.wrap_socket(socket.create_connection((host,443),timeout=6),
                             server_hostname=host) as s:
            cert = s.getpeercert()
            exp  = cert.get("notAfter","")
            if exp:
                days = (datetime.strptime(exp,"%b %d %H:%M:%S %Y %Z")-datetime.utcnow()).days
                if days < 0:
                    out.append(("SSL/TLS","Certificate EXPIRED","Critical",
                                f"Expired {abs(days)} days ago","Renew certificate immediately"))
                elif days < 30:
                    out.append(("SSL/TLS","Certificate expiring soon","High",
                                f"Expires in {days} days","Renew before expiry"))
            proto = s.version()
            if proto in ("TLSv1","TLSv1.1","SSLv2","SSLv3"):
                out.append(("SSL/TLS",f"Weak protocol ({proto})","High",
                            "Old TLS in use","Enforce TLS 1.2 or 1.3 only"))
    except ssl.SSLCertVerificationError:
        out.append(("SSL/TLS","Untrusted / self-signed certificate","Critical",
                    "Cert not trusted by browser","Use a cert from a trusted CA"))
    except (socket.timeout, OSError):
        out.append(("SSL/TLS","HTTPS unavailable (port 443 closed)","Critical",
                    "No HTTPS on this host","Enable HTTPS with a valid TLS cert"))
    except Exception: pass
    return out

def chk_cookies(resp):
    out = []
    for c in resp.cookies:
        if not c.secure:
            out.append(("Cookies",f"Cookie \'{c.name}\': Secure flag missing","Medium",
                        "Sent over plain HTTP","Add Secure flag"))
        raw = resp.headers.get("set-cookie","").lower()
        if c.name.lower() in raw and "httponly" not in raw:
            out.append(("Cookies",f"Cookie \'{c.name}\': HttpOnly missing","Medium",
                        "Readable by JS — XSS risk","Add HttpOnly flag"))
        if not c.get_nonstandard_attr("SameSite"):
            out.append(("Cookies",f"Cookie \'{c.name}\': SameSite not set","Low",
                        "CSRF risk","Set SameSite=Lax or Strict"))
    return out

def chk_server(resp):
    out, h = [], resp.headers
    s = h.get("Server","")
    if s and any(ch.isdigit() for ch in s):
        out.append(("Info Disclosure","Server version in header","Low",
                    f"Server: {s}","Remove version from Server header"))
    if h.get("X-Powered-By"):
        out.append(("Info Disclosure","X-Powered-By header exposes tech","Low",
                    f"X-Powered-By: {h[\'X-Powered-By\']}","Remove X-Powered-By header"))
    if h.get("X-AspNet-Version"):
        out.append(("Info Disclosure","ASP.NET version disclosed","Low",
                    h["X-AspNet-Version"],"Remove X-AspNet-Version in web.config"))
    return out

def chk_methods(url):
    out = []
    for m in ["TRACE","PUT","DELETE"]:
        try:
            r = requests.request(m, url, timeout=5, allow_redirects=False,
                                 headers={"User-Agent":"CyberSentry"})
            if m=="TRACE" and r.status_code==200:
                out.append(("HTTP Methods","TRACE method enabled — XST risk","Medium",
                            "Server echoes requests","Disable TRACE on server"))
            elif m in ("PUT","DELETE") and r.status_code not in (403,404,405,501):
                out.append(("HTTP Methods",f"{m} method accepted","Medium",
                            f"Server returned {r.status_code}",f"Block {m} in server config"))
        except: pass
    return out

def chk_paths(base):
    out = []
    paths = [
        ("/.git/config","Critical","Git config exposed","Block .git from web access"),
        ("/.env","Critical",".env file public","Move .env outside web root"),
        ("/phpinfo.php","High","PHP info page public","Delete phpinfo.php from server"),
        ("/admin","High","Admin panel reachable","Restrict /admin by IP + auth"),
        ("/backup","High","Backup folder accessible","Move backups outside web root"),
        ("/.htaccess","Medium",".htaccess accessible","Deny .htaccess in server config"),
        ("/wp-admin","Medium","WordPress admin visible","Restrict wp-admin by IP"),
        ("/server-status","Medium","Apache status page open","Restrict to localhost only"),
        ("/robots.txt","Informational","robots.txt reveals paths","Review Disallow entries"),
    ]
    for path, sev, msg, fix in paths:
        r, err = safe_get(base+path, timeout=5, redirects=False)
        if r is not None and r.status_code in (200,301,302):
            out.append(("Sensitive Paths", msg, sev,
                        f"\'{path}\' returned HTTP {r.status_code}", fix))
    return out

def chk_forms(resp, url):
    out = []
    try:
        soup = BeautifulSoup(resp.text, "html.parser")
        for i, form in enumerate(soup.find_all("form"), 1):
            if form.get("method","get").lower() == "post":
                inputs = [inp.get("name","").lower() for inp in form.find_all("input")]
                if not any("csrf" in n or "token" in n for n in inputs):
                    out.append(("Form Security",f"POST form #{i} — no CSRF token","High",
                                f"Action: {form.get(\'action\',\'/\')}","Add CSRF token to form"))
            for inp in form.find_all("input"):
                if inp.get("type","").lower()=="password":
                    if inp.get("autocomplete","on").lower() not in ("off","new-password"):
                        out.append(("Form Security","Password field has autocomplete on","Low",
                                    "Browser may save password","Add autocomplete=off"))
    except: pass
    return out

def chk_clickjack(resp):
    h   = resp.headers
    xfo = "X-Frame-Options" in h
    csp = "frame-ancestors" in h.get("Content-Security-Policy","").lower()
    if not xfo and not csp:
        return [("Clickjacking","Page embeddable in iframe","High",
                 "No X-Frame-Options or CSP frame-ancestors",
                 "Add X-Frame-Options: DENY")]
    return []

def chk_mixed(resp, url):
    if not url.startswith("https://"): return []
    out = []
    try:
        soup = BeautifulSoup(resp.text, "html.parser")
        mixed = []
        for tag, attr in [("script","src"),("img","src"),("link","href"),("iframe","src")]:
            for el in soup.find_all(tag):
                v = el.get(attr,"")
                if v.startswith("http://"): mixed.append(v[:60])
        if mixed:
            out.append(("Mixed Content",f"{len(mixed)} HTTP resource(s) on HTTPS page","Medium",
                        mixed[0],"Upgrade all resource URLs to HTTPS"))
    except: pass
    return out

def chk_redirect(url):
    for param in ["redirect","url","next","return","goto"]:
        try:
            r = requests.get(f"{url}?{param}=http://evil.example.com",
                             timeout=5, allow_redirects=False,
                             headers={"User-Agent":"CyberSentry"})
            if r.status_code in (301,302,303,307,308):
                if "evil.example.com" in r.headers.get("Location",""):
                    return [("Open Redirect","Open redirect found","High",
                             f"?{param}= follows external URLs",
                             "Whitelist allowed redirect destinations")]
        except: pass
    return []

#master Scan 
def run_scan(raw_url):
    url  = raw_url.strip()
    if not url.startswith(("http://","https://")): url = "https://"+url
    url  = url.rstrip("/")
    host = urlparse(url).hostname or ""
    all_findings, errors = [], []

    resp, err = safe_get(url)
    if err or resp is None:
        return [], [f"Cannot reach target: {err}"], url

    checks = [
        ("Security Headers",    lambda: chk_headers(resp)),
        ("SSL / TLS",           lambda: chk_ssl(host)),
        ("Cookies",             lambda: chk_cookies(resp)),
        ("Server Info Leakage", lambda: chk_server(resp)),
        ("HTTP Methods",        lambda: chk_methods(url)),
        ("Sensitive Paths",     lambda: chk_paths(url)),
        ("Form Security",       lambda: chk_forms(resp, url)),
        ("Clickjacking",        lambda: chk_clickjack(resp)),
        ("Mixed Content",       lambda: chk_mixed(resp, url)),
        ("Open Redirect",       lambda: chk_redirect(url)),
    ]

    bar    = st.progress(0, text="Initialising...")
    status = st.empty()

    for i, (name, fn) in enumerate(checks):
        status.markdown(f"\\U0001f50d **Checking {name}...** ({i+1}/{len(checks)})")
        try:
            all_findings.extend(fn())
        except Exception as e:
            errors.append(f"{name}: {e}")
        bar.progress((i+1)/len(checks), text=f"Done: {name}")

    bar.empty()
    status.empty()
    return all_findings, errors, url

#  like here iam using the Scoreing with Grade in this cell
def score(findings):
    if not findings: return pd.DataFrame(), 0, "A"
    df = pd.DataFrame(findings, columns=["Category","Finding","Severity","Detail","Fix"])
    df["Score"] = df["Severity"].map(SEV_SCORE).fillna(0).astype(int)
    total = int(df["Score"].sum())
    grade = "A" if total<=5 else "B" if total<=15 else "C" if total<=30 else "D" if total<=50 else "F"
    return df, total, grade

# here  it will send  the automatically mail 
def send_email(target, ts, total, grade, df):
    if not all([SENDER, PASSWORD, RECEIVER]):
        return False, "Email not configured in u.env"
    hc = df[df["Severity"].isin(["Critical","High"])]
    if hc.empty:
        return False, "No High/Critical — no email needed"

    rows = "".join(f"""
    <tr>
      <td style=\'padding:8px;border-bottom:1px solid #eee\'>{r.Finding}</td>
      <td style=\'padding:8px;border-bottom:1px solid #eee\'>
        <b style=\'color:{SEV_COLOR[r.Severity]}\'>{r.Severity}</b></td>
      <td style=\'padding:8px;border-bottom:1px solid #eee;text-align:center\'>{r.Score}</td>
      <td style=\'padding:8px;border-bottom:1px solid #eee;font-size:12px\'>{r.Fix}</td>
    </tr>""" for r in hc.itertuples())

    html = f"""
    <div style=\'font-family:Arial;max-width:650px;margin:0 auto\'>
      <div style=\'background:#1a202c;color:white;padding:24px;border-radius:8px 8px 0 0\'>
        <h2 style=\'margin:0\'>\\U0001f6e1\\ufe0f CyberSentry — Security Alert</h2>
        <p style=\'margin:6px 0 0;color:#a0aec0\'>Automated scan report</p>
      </div>
      <div style=\'background:#fff5f5;border-left:5px solid #e53e3e;padding:14px 20px\'>
        <b style=\'color:#c53030\'>\\u26a0\\ufe0f {len(hc)} High/Critical issue(s) found on {target}</b>
      </div>
      <div style=\'padding:20px;background:#f7fafc\'>
        <table style=\'width:100%;border-collapse:collapse\'>
          <tr>
            <td style=\'text-align:center;padding:10px\'>
              <div style=\'font-size:40px;font-weight:900;color:{GRADE_COL[grade]}\'>{grade}</div>
              <div style=\'font-size:11px;color:#718096\'>SECURITY GRADE</div>
            </td>
            <td style=\'text-align:center;padding:10px\'>
              <div style=\'font-size:40px;font-weight:900;color:#e53e3e\'>{total}</div>
              <div style=\'font-size:11px;color:#718096\'>RISK SCORE</div>
            </td>
            <td style=\'text-align:center;padding:10px\'>
              <div style=\'font-size:40px;font-weight:900;color:#dd6b20\'>{len(hc)}</div>
              <div style=\'font-size:11px;color:#718096\'>CRITICAL/HIGH</div>
            </td>
          </tr>
        </table>
      </div>
      <table style=\'width:100%;border-collapse:collapse;font-size:13px\'>
        <tr style=\'background:#1a202c;color:white\'>
          <th style=\'padding:10px;text-align:left\'>Finding</th>
          <th style=\'padding:10px;text-align:left\'>Severity</th>
          <th style=\'padding:10px;text-align:center\'>Score</th>
          <th style=\'padding:10px;text-align:left\'>Recommended Fix</th>
        </tr>{rows}
      </table>
      <table style=\'width:100%;font-size:13px;margin:16px 0\'>
        <tr><td style=\'color:#718096;padding:6px 20px;width:140px\'>Target</td>
            <td style=\'padding:6px;font-weight:bold\'>{target}</td></tr>
        <tr style=\'background:#f7fafc\'>
          <td style=\'color:#718096;padding:6px 20px\'>Scan time</td>
          <td style=\'padding:6px\'>{ts}</td></tr>
      </table>
      <div style=\'background:#edf2f7;padding:14px 20px;font-size:11px;color:#718096;
                  border-radius:0 0 8px 8px\'>
        <b>Automated disclaimer:</b> This report is generated by CyberSentry for
        authorised educational security assessment only. Scanning systems you do not
        own or have permission to test is illegal. Generated: {ts}
      </div>
    </div>"""

    worst = "CRITICAL" if "Critical" in hc["Severity"].values else "HIGH"
    msg   = MIMEMultipart("alternative")
    msg["Subject"] = f"\\U0001f6a8 [{worst}] CyberSentry — {len(hc)} issues on {urlparse(target).hostname}"
    msg["From"]    = SENDER
    msg["To"]      = RECEIVER
    msg.attach(MIMEText(html, "html"))

    try:
        with smtplib.SMTP("smtp.gmail.com", 587) as s:
            s.ehlo(); s.starttls(); s.login(SENDER, PASSWORD); s.send_message(msg)
        return True, f"Alert sent to {RECEIVER}"
    except smtplib.SMTPAuthenticationError:
        return False, "Gmail auth failed — check u.env credentials"
    except Exception as e:
        return False, str(e)

#Session state 
for k, v in {"done":False,"df":None,"score":0,"grade":"—",
             "target":"","ts":"","errors":[],"email":""}.items():
    if k not in st.session_state:
        st.session_state[k] = v
#side bar 
with st.sidebar:
    st.image("https://img.icons8.com/ios-filled/80/4f8ef7/shield.png", width=60)
    st.title("CyberSentry")
    st.caption("Web Vulnerability Scanner")
    st.divider()

    email_ok = bool(SENDER and PASSWORD and RECEIVER)
    st.subheader("\\u2699\\ufe0f Status")

    # BUG FIX 2: Use if/else block instead of ternary.
    # Ternary returns a DeltaGenerator object which Jupyter auto-prints
    # as that long ugly method list in your sidebar.
    if email_ok:
        st.success("\\u2705 Email alerts ON")
    else:
        st.warning("\\u26a0\\ufe0f Email not set in u.env")

    st.divider()
    st.subheader("\\U0001f3af Target")
    pick   = st.selectbox("Select test site", TEST_SITES)
    custom = st.text_input("Custom URL", placeholder="https://...") if "Custom" in pick else ""
    target_url = custom.strip() if "Custom" in pick else pick
    st.divider()

    scan_btn  = st.button("\\U0001f680 Start Scan", type="primary", use_container_width=True)
    clear_btn = st.button("\\U0001f5d1\\ufe0f Clear",               use_container_width=True)

    if clear_btn:
        for k in ["done","df","score","grade","target","ts","errors","email"]:
            st.session_state[k] = False if k=="done" else (None if k=="df" else "")
        st.rerun()

    st.divider()
    st.caption("\\u26a0\\ufe0f Only scan sites you are authorised to test.")

# Trigger scan 
if scan_btn:
    if not target_url:
        st.sidebar.error("Please enter a target URL.")
    else:
        findings, errors, final_url = run_scan(target_url)
        df, total, grade = score(findings)
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        st.session_state.update(dict(done=True, df=df, score=total, grade=grade,
                                     target=final_url, ts=ts, errors=errors))
        if not df.empty:
            sent, msg = send_email(final_url, ts, total, grade, df)
            st.session_state["email"] = msg
            if sent:
                st.toast("\\U0001f4e7 Alert email sent!", icon="\\u2705")
        else:
            st.session_state["email"] = "No High/Critical issues — no alert sent."

#  Landing page 
if not st.session_state["done"]:
    st.markdown("# \\U0001f6e1\\ufe0f CyberSentry")
    st.markdown("### Web Application Vulnerability Scanner")
    st.info("\\U0001f448 Pick a test site from the sidebar and click **Start Scan**")
    st.divider()
    cols = st.columns(5)
    items = [
        ("\\U0001f512","Security Headers","HSTS \\u00b7 CSP \\u00b7 X-Frame-Options \\u00b7 3 more"),
        ("\\U0001f4dc","SSL / TLS","Cert expiry \\u00b7 weak protocols \\u00b7 trust errors"),
        ("\\U0001f36a","Cookies","Secure \\u00b7 HttpOnly \\u00b7 SameSite flags"),
        ("\\U0001f4c2","Sensitive Paths",".git \\u00b7 .env \\u00b7 admin \\u00b7 phpinfo \\u00b7 5 more"),
        ("\\U0001f4cb","Forms & Logic","CSRF \\u00b7 open redirect \\u00b7 mixed content"),
    ]
    for col,(icon,title,desc) in zip(cols,items):
        with col:
            st.markdown(f"**{icon} {title}**")
            st.caption(desc)
    st.stop()

# Results page
df        = st.session_state["df"]
total     = st.session_state["score"]
grade     = st.session_state["grade"]
target    = st.session_state["target"]
ts        = st.session_state["ts"]
email_msg = st.session_state["email"]
gc        = GRADE_COL.get(grade,"#555")

st.markdown("# \\U0001f6e1\\ufe0f CyberSentry — Scan Report")
st.caption(f"\\U0001f310 **{target}** &nbsp;|&nbsp; \\U0001f550 **{ts}**")


if email_msg:
    if "sent" in email_msg:
        st.success(f"\\U0001f4e7 {email_msg}")
    else:
        st.info(f"\\U0001f4e7 {email_msg}")

# Stop here if df is empty — prevents all chart errors below
if df is None or df.empty:
    st.success("\\u2705 No vulnerabilities detected — site looks well secured!")
    st.stop()

# Scorecard 
st.markdown("---")
g0, g1, g2, g3, g4, g5 = st.columns([1.2,1,1,1,1,1])

with g0:
    st.markdown(f"""
    <div style=\'text-align:center;padding:10px\'>
      <div style=\'font-size:64px;font-weight:900;color:{gc};
                  border:5px solid {gc};border-radius:50%;
                  width:90px;height:90px;display:inline-flex;
                  align-items:center;justify-content:center\'>{grade}</div>
      <div style=\'margin-top:8px;font-size:22px;font-weight:800;color:{gc}\'>{total} pts</div>
      <div style=\'font-size:11px;color:#888\'>risk score</div>
    </div>""", unsafe_allow_html=True)

for col, sev, clr in [
    (g1,"Critical","#e53e3e"),(g2,"High","#dd6b20"),
    (g3,"Medium","#d69e2e"),(g4,"Low","#3182ce"),(g5,"Informational","#718096")]:
    cnt = int((df["Severity"] == sev).sum())
    with col:
        st.markdown(f"""
        <div style=\'border-top:4px solid {clr};border-radius:8px;
                    padding:14px;text-align:center;background:#f8f9fa\'>
          <div style=\'font-size:32px;font-weight:800;color:{clr}\'>{cnt}</div>
          <div style=\'font-size:12px;color:#555\'>{sev}</div>
        </div>""", unsafe_allow_html=True)

st.markdown("---")

#  Charts  
st.markdown("### \\U0001f4ca Risk Analysis")
c1, c2 = st.columns(2)

with c1:
    cats   = df["Category"].unique().tolist()
    scores = [int(df[df["Category"]==c]["Score"].sum()) for c in cats]
    if len(cats) >= 3:
        fig = go.Figure(go.Scatterpolar(
            r=scores+[scores[0]], theta=cats+[cats[0]],
            fill="toself",
            fillcolor="rgba(229,62,62,0.15)",
            line=dict(color="#e53e3e", width=2)))
        fig.update_layout(title="Risk by Category", height=350,
            polar=dict(radialaxis=dict(visible=True, range=[0,max(scores)+2])),
            showlegend=False, margin=dict(t=50,b=20,l=20,r=20))
        st.plotly_chart(fig, use_container_width=True)
    else:
        st.info("Need at least 3 categories for radar chart.")

with c2:
    sev_order  = ["Critical","High","Medium","Low","Informational"]
    sev_colors = ["#e53e3e","#dd6b20","#d69e2e","#3182ce","#718096"]
    counts     = df["Severity"].value_counts().reindex(sev_order).fillna(0).astype(int)
    fig2 = go.Figure(go.Funnel(
        y=sev_order, x=counts.values,
        textinfo="value+percent initial",
        marker=dict(color=sev_colors)))
    fig2.update_layout(title="Severity Breakdown", height=350,
                       margin=dict(t=50,b=10,l=10,r=10))
    st.plotly_chart(fig2, use_container_width=True)

c3, c4 = st.columns(2)
with c3:
    cat_df = df.groupby("Category").size().reset_index(name="Count").sort_values("Count")
    fig3   = px.bar(cat_df, x="Count", y="Category", orientation="h",
                    color="Count", color_continuous_scale="OrRd",
                    text="Count", title="Findings per Category")
    fig3.update_traces(textposition="outside")
    fig3.update_layout(height=320, showlegend=False,
                       yaxis=dict(categoryorder="total ascending"),
                       margin=dict(t=50,b=10,l=10,r=30))
    st.plotly_chart(fig3, use_container_width=True)

with c4:
    fig4 = px.treemap(df, path=["Category","Severity","Finding"],
                      values="Score",
                      color="Severity",
                      color_discrete_map={
                          "Critical":"#e53e3e","High":"#dd6b20",
                          "Medium":"#d69e2e","Low":"#3182ce","Informational":"#718096"},
                      title="Vulnerability Map")
    fig4.update_layout(height=320, margin=dict(t=50,b=5,l=5,r=5))
    st.plotly_chart(fig4, use_container_width=True)

# Findings list 
st.markdown("---")
st.markdown("### \\U0001f50d All Findings")

sf1, sf2 = st.columns(2)
with sf1:
    sev_f = st.multiselect("Severity filter",
        ["Critical","High","Medium","Low","Informational"],
        default=["Critical","High","Medium","Low","Informational"])
with sf2:
    cat_f = st.multiselect("Category filter",
        sorted(df["Category"].unique()), default=sorted(df["Category"].unique()))

filt = df[df["Severity"].isin(sev_f) & df["Category"].isin(cat_f)]
st.caption(f"Showing {len(filt)} of {len(df)} findings")

for _, row in filt.sort_values("Score", ascending=False).iterrows():
    clr = SEV_COLOR.get(row["Severity"],"#555")
    st.markdown(f"""
    <div style=\'border-left:4px solid {clr};border-radius:0 8px 8px 0;
                padding:12px 16px;margin:6px 0;background:#f8f9fa\'>
      <div style=\'display:flex;justify-content:space-between;align-items:center\'>
        <span style=\'font-weight:700;font-size:15px\'>{row["Finding"]}</span>
        <span style=\'background:{clr};color:white;padding:2px 10px;
                     border-radius:12px;font-size:11px;font-weight:700\'>{row["Severity"]}</span>
      </div>
      <div style=\'font-size:12px;color:#666;margin:3px 0\'>{row["Category"]} · Score: <b>{row["Score"]}</b></div>
      <div style=\'font-size:13px;color:#333;margin-top:4px\'>{row["Detail"]}</div>
      <div style=\'font-size:12px;background:#ebf8ff;border-left:3px solid #3182ce;
                  padding:5px 8px;margin-top:6px;border-radius:0 4px 4px 0;color:#2b6cb0\'>
        \\U0001f4a1 {row["Fix"]}</div>
    </div>""", unsafe_allow_html=True)

# Export 
st.markdown("---")
st.markdown("### \\U0001f4be Export Results")
e1, e2 = st.columns(2)
host   = urlparse(target).hostname or "scan"
dstr   = datetime.now().strftime("%Y%m%d_%H%M")
with e1:
    st.download_button("\\u2b07\\ufe0f Full Report CSV",
        df.to_csv(index=False).encode(),
        f"cybersentry_{host}_{dstr}.csv","text/csv",use_container_width=True)
with e2:
    st.download_button("\\u2b07\\ufe0f Filtered CSV",
        filt.to_csv(index=False).encode(),
        f"cybersentry_filtered_{dstr}.csv","text/csv",use_container_width=True)

if st.session_state["errors"]:
    with st.expander(f"\\u26a0\\ufe0f {len(st.session_state[\'errors\'])} scan warning(s)"):
        for e in st.session_state["errors"]:
            st.caption(f"• {e}")
'''

# Write app.py
with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code.strip())

import os
lines = len(open("app.py", encoding="utf-8").readlines())
size  = os.path.getsize("app.py")
print(f"app.py  ✅  {lines} lines  |  {size:,} bytes")

# Write Streamlit dark theme config
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write("""
[theme]
primaryColor             = "#4f8ef7"
backgroundColor          = "#0f1117"
secondaryBackgroundColor = "#1a1d2e"
textColor                = "#e2e8f0"
font                     = "sans serif"
""")
print(".streamlit/config.toml  ✅")

# Create u.env only if it doesn't already exist (keeps your saved credentials)
if not os.path.exists("u.env"):
    with open("u.env", "w") as f:
        f.write("# Fill in your Gmail credentials below\n")
        f.write("GMAIL_SENDER=your_email@gmail.com\n")
        f.write("GMAIL_PASSWORD=xxxx xxxx xxxx xxxx\n")
        f.write("GMAIL_RECIPIENT=recipient@example.com\n")
    print("u.env  ✅  (template created — fill in your credentials)")
else:
    print("u.env  ✅  (already exists — credentials kept)")

print()
print("✅  Setup complete!  Run Cell 3 to launch the dashboard.")


# ============================================================
#  CELL 3 — Launch Streamlit  (run every time)
# ============================================================

import subprocess, sys, time

print("🚀  Starting CyberSentry...")

subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.headless", "true",
     "--server.port", "8501"],
)

time.sleep(3)
print()
print("✅  Dashboard is running!")
print("👉  Open this in your browser  →  http://localhost:8501")
print()
print("Test targets:")
print("   • http://testphp.vulnweb.com")
print("   • http://testasp.vulnweb.com")
print("   • http://testhtml5.vulnweb.com")
print("   • http://zero.webappsecurity.com")
print()
print("📧  Email alerts fire automatically on High/Critical findings.")
print("    Make sure credentials are saved in u.env")

✅  streamlit
✅  requests
✅  pandas
✅  plotly
✅  beautifulsoup4
✅  python-dotenv

✅  All done — run Cell 2 to write app.py!
app.py  ✅  587 lines  |  27,061 bytes
.streamlit/config.toml  ✅
u.env  ✅  (already exists — credentials kept)

✅  Setup complete!  Run Cell 3 to launch the dashboard.
🚀  Starting CyberSentry...

✅  Dashboard is running!
👉  Open this in your browser  →  http://localhost:8501

Test targets:
   • http://testphp.vulnweb.com
   • http://testasp.vulnweb.com
   • http://testhtml5.vulnweb.com
   • http://zero.webappsecurity.com

📧  Email alerts fire automatically on High/Critical findings.
    Make sure credentials are saved in u.env
